In [1]:
pip install openrouteservice

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import geopandas as gpd
import os
import openrouteservice
import time
import folium

This script serves the purpose of generating a nxn matrix where n = number of planning units. We seek to calculate the travel time driving distance from each planning unit's centroid to the centroid of every other planning unit.

In [32]:
class distanceMatrix:
    def __init__(self, pu_path, key):
        self.pu_path = pu_path
        self.key = key
        self.client = openrouteservice.Client(key = key)

    def load_data(self):
        self.pu = gpd.read_file(f'{os.getcwd()}/data/{self.pu_path}').to_crs('EPSG:4326')

    def get_pu_centroids(self):
        self.pu['centroid'] = self.pu.geometry.centroid
        self.centroids = self.pu['centroid']

    def isochrone_branching(self, pu = 1, time = 100):      #isochrone branches for pu argument
        centroid = self.centroids.iloc[pu-1]
        coords = (centroid.x,centroid.y)
        self.iso = self.client.isochrones(
            locations = [coords],
            profile = 'driving-car',
            range = [time]
        )      

        self.iso_geo = gpd.GeoDataFrame.from_features(self.iso['features'], crs = 'EPSG:4326')['geometry']

    def distance_array(self, pu = 1, max_time = 3600, step = 60):
        N = len(self.centroids)
        self.times = [None] * N
        for t in range(step, max_time + step, step):                #branch isochrones by step
            self.isochrone_branching(pu = pu, time = t)
            isochrone = self.iso_geo.unary_union            #unary_union -> union_all?

            #get what centroids lay within isochrone
            overlap = self.centroids[self.centroids.within(isochrone)]
            indices = overlap.index.to_list()

            #enter new times into list
            for idx in indices:
                if self.times[idx] is None:
                    self.times[idx] = int(t/60)
            
            #break if all centroids have been reached
            if all(time is not None for time in self.times):
                break

            #sleep to avoid surpassing 20/minute quota
            time.sleep(3)
            
        
        
        
        
        

        
        

In [8]:
matrix = distanceMatrix('pu_2324_SPLIT.geojson', 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImRmZGMwZDA3NDVhYzRkNzY5Y2UzN2Q1YTk3MmNlNWQzIiwiaCI6Im11cm11cjY0In0=')
matrix.load_data()
matrix.get_pu_centroids()
matrix.isochrone_branching()
matrix.distance_array()

C:\Users\olubl\AppData\Local\Temp\ipykernel_7044\3963318138.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  self.pu['centroid'] = self.pu.geometry.centroid
C:\Users\olubl\AppData\Local\Temp\ipykernel_7044\3963318138.py:30: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  isochrone = self.iso_geo.unary_union            #unary_union -> union_all?
C:\Users\olubl\anaconda3\envs\spatialdata\Lib\site-packages\openrouteservice\client.py:211: UserWarning: Rate limit exceeded. Retrying for the 1st time.
  warnings.warn('Rate limit exceeded. Retrying for the {0}{1} time.'.format(retry_counter + 1,
C:\Users\olubl\anaconda3\envs\spatialdata\Lib\site-packages\openrouteservice\client.py:211: UserWarning: Rate limit exceeded. Retrying for the 2nd time.
  warnings.warn('Rate limit exceeded. Retrying for t

In [31]:
m = folium.Map((35.900,-78.882), tiles="OpenStreetMap", zoom_start = 10)
folium.GeoJson(matrix.iso_geo).add_to(m)
m

In [29]:
times = [int(tt/60) if tt is not None else None for tt in matrix.times]

In [30]:
times

[30,
 28,
 43,
 57,
 32,
 47,
 49,
 37,
 36,
 43,
 34,
 39,
 40,
 47,
 36,
 30,
 31,
 32,
 26,
 44,
 44,
 21,
 26,
 20,
 31,
 28,
 28,
 32,
 27,
 29,
 28,
 36,
 35,
 38,
 39,
 39,
 46,
 36,
 44,
 49,
 50,
 25,
 24,
 24,
 25,
 36,
 42,
 45,
 46,
 22,
 16,
 14,
 13,
 21,
 25,
 21,
 13,
 15,
 23,
 20,
 21,
 17,
 17,
 23,
 26,
 23,
 29,
 26,
 25,
 26,
 31,
 27,
 30,
 28,
 27,
 36,
 33,
 34,
 40,
 36,
 37,
 37,
 38,
 41,
 21,
 24,
 26,
 24,
 25,
 24,
 33,
 33,
 30,
 38,
 36,
 33,
 46,
 43,
 43,
 41,
 40,
 41,
 46,
 46,
 44,
 47,
 53,
 46,
 50,
 52,
 49,
 42,
 46,
 45,
 42,
 43,
 44,
 41,
 43,
 43,
 42,
 43,
 40,
 42,
 42,
 43,
 43,
 42,
 47,
 42,
 26,
 38,
 42,
 42,
 44,
 45,
 43,
 42,
 43,
 45,
 31,
 28,
 27,
 23,
 29,
 29,
 38,
 33,
 40,
 44,
 46,
 47,
 39,
 35,
 20,
 12,
 14,
 42,
 37,
 40,
 29,
 25,
 26,
 19,
 6,
 47,
 51,
 None,
 51,
 44,
 47,
 41,
 43,
 45,
 36,
 35,
 26,
 18,
 25,
 40,
 36,
 36,
 28,
 52,
 24,
 20,
 26,
 18,
 26,
 26,
 12,
 14,
 18,
 18,
 19,
 20,
 22,
 23,
 22,
 29,